# Implementing a RAG System

## My knowledge base

I selected climate change as the topic for my knowledge base. I used these three articles about its impacts:

- [NOAA: Climate Change Impacts](https://www.noaa.gov/education/resource-collections/climate/climate-change-impacts)
- [NRDC: What Are the Effects of Climate Change?](https://www.nrdc.org/stories/what-are-effects-climate-change)
- [National Geographic: Global Warming Effects](https://www.nationalgeographic.com/environment/article/global-warming-effects)

## Implementation Requirements  

1. Document Processing and Indexing
    - Process documents into suitable chunks for retrieval
    - Generate embeddings for each chunk using an appropriate model
    - Create a simple vector index for similarity search

In [11]:
import requests
from bs4 import BeautifulSoup

# Function to fetch and extract paragraph text from a webpage
def fetch_article_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find all paragraphs on the page
    paragraphs = soup.find_all('p')
    
    # Join them into one big string
    text = ' '.join([para.get_text() for para in paragraphs])
    return text

# Article URLs (your 3 sources)
urls = [
    "https://www.noaa.gov/education/resource-collections/climate/climate-change-impacts",
    "https://www.nrdc.org/stories/what-are-effects-climate-change",
    "https://www.nationalgeographic.com/environment/article/global-warming-effects"
]

# Extract the text of all articles
documents = [fetch_article_text(url) for url in urls]

I used web scraping to collect text from the three articles I selected. With `requests`, I fetched each webpage, then used **BeautifulSoup** from the `bs4` library to parse the HTML. My `fetch_article_text` function extracts the paragraphs (`<p>` elements) and joins their text into one string per article.

In [ ]:
# Group sentences into chunks for retrieval
def chunk_text(text, max_length=500):
    sentences = text.split('. ')
    chunks = []
    current_chunk = ''
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_length:
            current_chunk += sentence + '. '
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + '. '
    
    # Add the last chunk if it's not empty
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks

# Split all articles into small chunks for retrieval
all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc))

I split each article into sentence-based chunks to make the text easier to retrieve. My `chunk_text` function groups sentences up to about 500 characters and keeps sentence boundaries where possible. The implementation does not add overlap between chunks, so context at a boundary may be lost. I applied this function to all three articles and combined the resulting chunks in `all_chunks`.

In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a pre-trained model for turning text into vector embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate vector representations of each chunk
embeddings = model.encode(all_chunks)

# Create a FAISS index to store and search embeddings
dimension = embeddings.shape[1]  # e.g., 384
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

I used a pre-trained SentenceTransformer model to convert each text chunk into a numerical embedding that represents its meaning. I then stored those embeddings in a FAISS index so I could search for passages by semantic similarity.

2. Retrieval Component
    - Implement a function to convert user queries into embeddings
    - Develop a retrieval mechanism to find relevant documents based on similarity
    - Include a method to select the top-k most relevant passages

In [14]:
# Given a user query, find the top-k most similar chunks
def retrieve(query, top_k=3):
    query_embedding = model.encode([query])
    
    # Search the FAISS index
    distances, indices = index.search(np.array(query_embedding), top_k)
    
    # Return the most relevant chunks
    return [all_chunks[i] for i in indices[0]]

I wrote `retrieve` to find the chunks most relevant to a question. The function converts the question into an embedding with the same model, searches the FAISS index and returns the three closest chunks by default.

3. Generation with Context
    - Create a prompt template that incorporates retrieved passages
    - Use an accessible language model (e.g., Gemini via API, or a local model like Llama-2)
    - Generate answers that reference the retrieved information


In [ ]:
from transformers import pipeline

# Load a small open-source text generation model (e.g., GPT-2)
generator = pipeline("text-generation", model="gpt2")

# Given a query and retrieved context, generate an answer
def generate_answer(query, context_chunks):
    context = '\n'.join(context_chunks)
    
    # Prompt template that combines context with the query
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Generate text using the model
    response = generator(prompt, max_length=200, num_return_sequences=1)
    
    # Extract the generated answer from the model output
    return response[0]['generated_text'].split("Answer:")[-1].strip()

Device set to use mps:1


I used a pre-trained GPT-2 text-generation pipeline to generate an answer from the retrieved passages. In `generate_answer`, I combine the context chunks and the question in a prompt, send it to the model and extract the generated text after `Answer:`.

4. Evaluation
    - Test your system with at least 10 different queries
    - Measure performance using appropriate metrics (relevance of retrieved passages, accuracy of generated answers)
    - Compare results

In [18]:
test_queries = [
    "What are the effects of global warming?",
    "How does climate change affect weather?",
    "What happens to human health as climate changes?",
    "What are some impacts on agriculture?",
    "How does climate change affect the oceans?",
    "What are long-term climate change consequences?",
    "What are extreme weather events caused by global warming?",
    "How does climate change affect animals?",
    "What are economic effects of climate change?",
    "Can climate change impact water supply?"
]
for query in test_queries:
    print(f"\nQ: {query}")
    retrieved = retrieve(query)
    print("→ Retrieved Passages:")
    for i, chunk in enumerate(retrieved):
        print(f"  [{i+1}] {chunk[:200]}...")  # Print first 200 characters
    print("→ Answer:", generate_answer(query, retrieved))


Q: What are the effects of global warming?


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


→ Retrieved Passages:
  [1] Sources of this gas include agriculture (mostly livestock), leaks from oil and gas production, and waste from landfills.  One of the most concerning impacts of global warming is the effect warmer temp...
  [2] The results of this review is synthesized in regularly published reports known as the Intergovernmental Panel on Climate Change (IPCC) reports.  A comprehensive report outlines how disruptive a global...
  [3] All of these changes are emerging as humans continue to add heat-trapping greenhouse gases, like carbon dioxide and methane, to the atmosphere. When fossil fuel emissions are pumped into the atmospher...


ValueError: Input length of input_ids is 200, but `max_length` is set to 200. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.

I prepared ten climate-related questions covering topics such as weather, health, agriculture, oceans and water supply. For each question, I retrieve the three closest chunks and pass them to the generator. This gives me examples to inspect, but it does not by itself measure retrieval or answer quality.

In [ ]:
def baseline_answer(query):
    prompt = f"Question: {query}\nAnswer:"
    response = generator(prompt, max_new_tokens=100)
    return response[0]['generated_text'].split("Answer:")[-1].strip()
for query in test_queries:
    rag = generate_answer(query, retrieve(query))
    baseline = baseline_answer(query)
    print(f"\nQ: {query}")
    print("→ Baseline Answer:", baseline)
    print("→ RAG Answer     :", rag)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Q: What are the effects of global warming?
→ Baseline Answer: Global warming is an effect on ecosystems and human health. However, it is unclear how much global warming actually will affect the food supply in the United States, or in other parts of the world. The United States is projected to lose a significant share of its food security, including significant portion on energy products and services. An average of 20% of the United States population lives in developing countries that are at very high risk of food insecurity. The resulting scarcity of food leads to large-scale food insecurity and
→ RAG Answer     : Although the IPCC is not yet able to definitively say exactly how much the climate will change, it predicts that it will increase or decrease.  This is because climate change changes the way the Earth's climate affects other things like rainfall and temperatures, and is often thought to be an environmental disaster.
The most important impact of global warming is an increase 

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Q: How does climate change affect weather?
→ Baseline Answer: The weather in South Africa is far darker than the other six countries in the region. The weather is colder than in the continental US.
Source: Weather.net
→ RAG Answer     : Climate change is changing the climate.

Scientists, especially those from the scientific community, tend to focus on two parts of the picture: the effects of global warming on snow and ice cover and the effects of climate change on precipitation and precipitation patterns.  These two aspects can be viewed separately from each other in the context of climate change.

The first part of the picture is much more complex.  The climate change caused by climate change has been known for decades, but it has not been


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...


Q: How does climate change affect animals?
→ Baseline Answer: I believe there are several major processes and interactions that affect the physical and behavioral changes of animals. However I do not accept that the amount of animal growth that has occurred over the past 200,000 years is the single biggest impact. It is not just about growing big or spreading the disease; much larger changes occur over time under these conditions.
Figure 7 shows the total number of species that have multiplied or decreased in size over time by a factor of 2, or approximately 2 billion years or more.
→ RAG Answer     : Because they are all mammals or birds.  Because they are all mammals or birds.  Because they are all mammals or birds.  Because they are all mammals or birds.  Because they are all mammals or birds.  Because they are all mammals or birds.
 Because they are all mammals or birds.  Because they are all mammals or birds.  Because they are all mammals or birds.  Because they are all mammals o

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Q: What are economic effects of climate change?
→ Baseline Answer: Many economists have argued that climate change is an event that has a major bearing on aggregate economic activity. These studies find that economic activity declines in response to climate change and that those who experience the declines have a larger income share of that income, thus supporting the notion that economic activity declines. The argument is that a decline in income or job opportunities would tend to cause a bigger decline in overall economic activity.
Another important analysis, conducted at the University of Iowa, concluded: "These data from these two
→ RAG Answer     : A major economic cost to humanity is global warming.  A significant cost to society is that of global warming, especially in the short term.  A major economic cost is that of global warming, particularly in the short term.  A major economic cost to society is that of human-induced climate change.  A substantial amount of that costs wil

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Q: Can climate change impact water supply?
→ Baseline Answer: No such consequences. When water is depleted (in large part due to fossil fuel consumption), it is released by large swaths of the land in the area (by runoff and from agriculture). This runoff, as part of the climate change, will have greater and stronger impacts on water usage than the rest.
What is the impact of water use on sea level rise (and associated water quality and depletion)?
What is the impact of changing climate on water use?
Where can I go to learn
→ RAG Answer     : The answer to this question is yes, but the answer is complicated.  Water supply is critical to any ecosystem's survival, so it is important to understand how much water we are getting from the atmosphere and how much it's being used to build or store. 

In order to understand how much water we are getting, our first question must be: How much water does it take to build something?  This is the question that many people have about how much we're 

I compared two approaches: a baseline that answers without retrieved context and a RAG version that uses the passages returned by `retrieve`. I ran both approaches on the same ten questions so I could inspect their responses side by side.

## Analysis of Retrieval Quality and Its Effect on Generation

In my written analysis of the test examples, I noted that some retrieved passages were relevant, while others were too general, incomplete or off topic. I found that these cases could lead to vague or repetitive answers. My chunks are grouped by sentence and do not overlap, so useful context may be split across chunk boundaries. A more careful chunking strategy or a reranker could help me investigate these retrieval failures.

## Discussion of Factuality Compared to Pure Generation

In my written comparison, I judged some RAG answers to be more grounded in the source passages than the baseline GPT-2 answers. However, I did not evaluate them against reference answers or calculate factuality metrics, so I cannot demonstrate a measured improvement. Retrieval also does not guarantee that a generated answer is correct: weak or ambiguous passages can still lead to speculative or inconsistent output. I treat this comparison as exploratory rather than as evidence that the system reliably reduces hallucinations.

## Error Analysis and Potential Improvements

I identified several limitations in my prototype. My chunking groups sentences to a target length but does not create the overlap described in the earlier analysis; this can separate context that belongs together. My generator uses GPT-2, which can struggle with coherent, context-grounded answers, and my prompt could give clearer instructions about how to use the retrieved passages. To rerun the notebook, I need an internet connection to fetch the articles and download the pretrained models, as well as `requests`, `beautifulsoup4`, `sentence-transformers`, `faiss-cpu`, `numpy` and `transformers` installed. I did not pin dependency versions, so the environment is not fully reproducible. To improve the system, I could refine the chunking, add a reranker, use a stronger language model and evaluate retrieval and answer quality separately on a labelled test set.